# 23 — Quantum State Preparation

Consolidates legacy experiments 68, 69, 70, 71, 72, 73, 74, 75, 76, and 77.

1. **SNAP calibration data generation** — produce SNAP gate parameter sweeps (Exp 68)
2. **SNAP gate optimization** — run optimizer for SNAP angles (Exp 69)
3. **SNAP verification via Wigner** — verify SNAP with Wigner tomography (Exp 70)
4. **ECD state preparation** — Echoed Conditional Displacement (Exp 71)
5. **ECD Wigner verification** — Wigner tomography of ECD-prepared state (Exp 72)
6. **Grape pulse state prep** — optimal-control pulse applied (Exp 73)
7. **Grape Wigner verification** — Wigner of Grape-prepared state (Exp 74)
8. **Selective number-dependent rotation** — SNAP + displacement combo (Exp 75)
9. **Displaced Fock state** — prepare |α, n⟩ (Exp 76)
10. **Cat state** — prepare Schrödinger cat state (Exp 77)

## 1. Shared Session Bootstrap

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from qualang_tools.units import unit

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "qubox").exists():
    REPO_ROOT = Path(r"E:\qubox")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from qubox.compat.notebook import (
    SNAPOptimization,
    StorageWignerTomography,
    fit_quality_gate,
    load_stage_checkpoint,
    open_notebook_stage,
    preview_or_apply_patch_ops,
    save_stage_checkpoint,
)

REGISTRY_BASE = Path(r"E:\qubox")
SAMPLE_ID = "post_cavity_sample_A"
COOLDOWN_ID = "cd_2025_02_22"
QOP_IP = "10.157.36.68"
CLUSTER_NAME = "Cluster_2"

stage = open_notebook_stage(
    stage_name="23_quantum_state_preparation",
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    qop_ip=QOP_IP,
    cluster_name=CLUSTER_NAME,
    force_reopen=True,
    close_existing=True,
)
session = stage.session
attr = stage.attr
SESSION_BOOTSTRAP_PATH = stage.bootstrap_path
u = unit(coerce_to_integer=True)

fock_checkpoint = load_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="22_fock_resolved_experiments",
)

print(f"Repo root on sys.path: {REPO_ROOT}")
print(f"Shared session bootstrap: {SESSION_BOOTSTRAP_PATH}")
print(f"Stage checkpoint path: {stage.checkpoint_path}")
print(f"QM endpoint: {QOP_IP} ({CLUSTER_NAME})")
if fock_checkpoint is not None:
    print(f"Prior stage 22 status: {fock_checkpoint['status']}")

## 2. State Preparation Defaults

In [ ]:
APPLY_SNAP_CALIBRATION = True

# --- SNAP (Exp 68, 69) ---
SNAP_MAX_N = 5
SNAP_N_AVG = 10000

# --- Wigner verification (Exp 70, 72, 74) ---
WIGNER_ALPHA_MAX = 3.0
WIGNER_ALPHA_STEP = 0.1
WIGNER_N_AVG = 10000

# --- ECD (Exp 71) ---
ECD_N_AVG = 10000

# --- General state prep ---
STATE_PREP_N_AVG = 10000

print("State preparation settings:")
print(f"  SNAP max_n: {SNAP_MAX_N}, n_avg: {SNAP_N_AVG}")
print(f"  Wigner: α_max={WIGNER_ALPHA_MAX}, dα={WIGNER_ALPHA_STEP}")
print(f"  ECD n_avg: {ECD_N_AVG}")
print(f"  General state prep n_avg: {STATE_PREP_N_AVG}")

## 3. SNAP Calibration Data — Exp 68

Sweep SNAP gate parameters to generate calibration data.

In [ ]:
snap_data_result = session.snap_calibration_sweep(
    max_n=SNAP_MAX_N,
    n_avg=SNAP_N_AVG,
)

print("SNAP calibration data generation complete.")

## 4. SNAP Optimization — Exp 69

Optimize SNAP gate angles using calibration data.

In [ ]:
snap_opt = SNAPOptimization(session)
snap_opt_result = snap_opt.run(n_avg=SNAP_N_AVG)
snap_opt_analysis = snap_opt.analyze(snap_opt_result, update_calibration=True)
snap_opt.plot(snap_opt_analysis)

snap_fit_ok = fit_quality_gate(snap_opt_analysis, r_squared_min=0.80)

if snap_fit_ok:
    snap_patch, _, snap_apply = preview_or_apply_patch_ops(
        session,
        reason="SNAP gate optimization",
        proposed_patch_ops=snap_opt_analysis.metadata.get("proposed_patch_ops", []),
        apply=APPLY_SNAP_CALIBRATION,
    )
    if snap_apply is not None:
        context_snapshot = getattr(session, "context_snapshot", None)
        attr = context_snapshot() if callable(context_snapshot) else getattr(session, "attributes", None)
else:
    print("WARNING: SNAP optimization fit quality gate FAILED — skipping apply.")

print("SNAP optimization complete.")

## 5. SNAP Verification via Wigner — Exp 70

Verify SNAP gate by measuring the Wigner function of the resulting state.

In [ ]:
wigner_snap = StorageWignerTomography(session)
wigner_snap_result = wigner_snap.run(
    alpha_max=WIGNER_ALPHA_MAX,
    alpha_step=WIGNER_ALPHA_STEP,
    n_avg=WIGNER_N_AVG,
    state_prep="snap",
)
wigner_snap_analysis = wigner_snap.analyze(wigner_snap_result, update_calibration=False)
wigner_snap.plot(wigner_snap_analysis)

print("SNAP Wigner verification complete.")

## 6. ECD State Preparation — Exp 71

Echoed Conditional Displacement (ECD) state preparation.

In [ ]:
ecd_result = session.ecd_state_preparation(
    n_avg=ECD_N_AVG,
)

print("ECD state preparation complete.")

## 7. ECD Wigner Verification — Exp 72

Wigner function of ECD-prepared state.

In [ ]:
wigner_ecd = StorageWignerTomography(session)
wigner_ecd_result = wigner_ecd.run(
    alpha_max=WIGNER_ALPHA_MAX,
    alpha_step=WIGNER_ALPHA_STEP,
    n_avg=WIGNER_N_AVG,
    state_prep="ecd",
)
wigner_ecd_analysis = wigner_ecd.analyze(wigner_ecd_result, update_calibration=False)
wigner_ecd.plot(wigner_ecd_analysis)

print("ECD Wigner verification complete.")

## 8. Grape Pulse State Prep — Exp 73

Apply numerically-optimized (GRAPE) control pulse to prepare a target state.

In [ ]:
grape_result = session.grape_state_preparation(
    n_avg=STATE_PREP_N_AVG,
)

print("GRAPE state preparation complete.")

## 9. Grape Wigner Verification — Exp 74

Wigner function of GRAPE-prepared state.

In [ ]:
wigner_grape = StorageWignerTomography(session)
wigner_grape_result = wigner_grape.run(
    alpha_max=WIGNER_ALPHA_MAX,
    alpha_step=WIGNER_ALPHA_STEP,
    n_avg=WIGNER_N_AVG,
    state_prep="grape",
)
wigner_grape_analysis = wigner_grape.analyze(wigner_grape_result, update_calibration=False)
wigner_grape.plot(wigner_grape_analysis)

print("GRAPE Wigner verification complete.")

## 10. Selective Number-Dependent Rotation — Exp 75

Apply SNAP + displacement combination for selective rotation.

In [ ]:
sndr_result = session.selective_number_dependent_rotation(
    n_avg=STATE_PREP_N_AVG,
)

print("Selective number-dependent rotation complete.")

## 11. Displaced Fock State |α, n⟩ — Exp 76

Prepare a displaced Fock state and verify via Wigner.

In [ ]:
disp_fock_result = session.displaced_fock_state_preparation(
    n_avg=STATE_PREP_N_AVG,
)

wigner_dfock = StorageWignerTomography(session)
wigner_dfock_result = wigner_dfock.run(
    alpha_max=WIGNER_ALPHA_MAX,
    alpha_step=WIGNER_ALPHA_STEP,
    n_avg=WIGNER_N_AVG,
    state_prep="displaced_fock",
)
wigner_dfock_analysis = wigner_dfock.analyze(wigner_dfock_result, update_calibration=False)
wigner_dfock.plot(wigner_dfock_analysis)

print("Displaced Fock state preparation + Wigner complete.")

## 12. Cat State — Exp 77

Prepare a Schrödinger cat state |α⟩ + |−α⟩ and verify via Wigner.

In [ ]:
cat_result = session.cat_state_preparation(
    n_avg=STATE_PREP_N_AVG,
)

wigner_cat = StorageWignerTomography(session)
wigner_cat_result = wigner_cat.run(
    alpha_max=WIGNER_ALPHA_MAX,
    alpha_step=WIGNER_ALPHA_STEP,
    n_avg=WIGNER_N_AVG,
    state_prep="cat",
)
wigner_cat_analysis = wigner_cat.analyze(wigner_cat_result, update_calibration=False)
wigner_cat.plot(wigner_cat_analysis)

print("Cat state preparation + Wigner complete.")

## 13. Save Checkpoint

In [ ]:
snap_applied = "snap_apply" in dir() and snap_apply is not None

stage_checkpoint_path = save_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="23_quantum_state_preparation",
    status="calibrated" if snap_applied else "characterized",
    summary="SNAP optimization, ECD, GRAPE, displaced Fock, and cat state preparation with Wigner verification.",
    consumed_inputs={
        "snap_max_n": SNAP_MAX_N,
        "snap_n_avg": SNAP_N_AVG,
        "wigner_alpha_max": WIGNER_ALPHA_MAX,
        "wigner_alpha_step": WIGNER_ALPHA_STEP,
        "ecd_n_avg": ECD_N_AVG,
        "state_prep_n_avg": STATE_PREP_N_AVG,
    },
    persisted_outputs={
        "snap_applied": snap_applied,
    },
    advisory_outputs={},
    next_stage="24_free_evolution_tomography",
    notes=[
        "SNAP calibration is the only applied calibration; all else is characterization.",
        "ECD, GRAPE, and direct state prep methods depend on correct displacement ops (notebook 08).",
    ],
    metrics={},
)

print(f"Stage checkpoint saved to: {stage_checkpoint_path}")